# Otimização de Prompts com GEPA + Kontex — HotPotQA

Este notebook demonstra o workflow de otimização de prompts usando o **GEPA** 
(Genetic Pareto Prompt Optimizer) integrado ao **Kontex** para responder 
questões do benchmark **HotPotQA**.

O agente otimizado é o `description_building`, cujo role (system prompt) é
evoluído pelo GEPA para que a conversa multi-agente produza respostas mais
precisas e completas.

## Visão geral

```
HotPotQA (questões 'hard')
        ↓
  Simulação EDD (Kontex)
  → especialistas com fragmentos de conhecimento
        ↓
  Conversa multi-agente
  questioning → description_building → self_critique
        ↓
  GEvalMetric (acurácia factual + completude)
        ↓
  GEPA otimiza o role do description_building
```

## 1. Configuração de paths e imports

In [9]:
import sys
import os
from pathlib import Path

# Raiz do repositório kontex_gepa
root_dir = Path(os.getcwd()).parent.resolve()
parent_dir = root_dir.parent

# Adiciona kontex/src, gepa/src e a raiz ao path
sys.path.insert(0, str(root_dir))
sys.path.insert(0, str(parent_dir / "kontex" / "src"))
sys.path.insert(0, str(parent_dir / "gepa" / "src"))

# Banco de dados persistente (deve ser definido antes de importar kontex)
os.environ["DATABASE_URL"] = f"sqlite:///{root_dir}/kontex_gepa_data.db"

print(f"root_dir : {root_dir}")
print(f"parent_dir: {parent_dir}")

root_dir : /home/kunumi/projetos/kontex_gepa
parent_dir: /home/kunumi/projetos


## 2. Importações do módulo kontex_gepa

In [10]:
from kontex_gepa import (
    EnvConfig,
    KontexFlowGeneralized,
    GEvalMetric,
    GEVAL_CRITERIA_HOTPOTQA,
    generate_hotpot_pareto_dataset,
)

from gepa import GEPAOptimizer, GEPAConfig
from gepa.core.system import CompoundAISystem, LanguageModule, IOSchema
from gepa.inference.factory import InferenceFactory
from gepa.config import InferenceConfig, OptimizationConfig, DatabaseConfig, ObservabilityConfig
from gepa.evaluation.base import SimpleFeedbackEvaluator

from kontex.llm.agents.hotpotqa_agents import hotpotqa_description_building_role
from kontex.simulation.edd.general_knowledge import FullKnowledge

print("✅ Importações concluídas.")

ImportError: cannot import name 'GEVAL_CRITERIA_HOTPOTQA' from 'kontex_gepa' (/home/kunumi/projetos/kontex_gepa/kontex_gepa.py)

## 3. Carregamento das variáveis de ambiente

In [ ]:
env = EnvConfig(env_file=str(root_dir / ".env"))

if not env.api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY não encontrado. "
        "Certifique-se de que o arquivo .env existe e contém a chave."
    )

print(f"API key: {'*' * 8}{env.api_key[-4:]}")
print(f"Base URL: {env.base_url}")
print(f"Model: {env.model}")

## 4. Geração do dataset HotPotQA

Cada datapoint corresponde a uma questão HotPotQA de nível 'hard'.
A simulação EDD distribui o contexto da questão entre especialistas simulados.

In [ ]:
N_QUESTIONS = 3   # número de questões HotPotQA
PARETO_SIZE = 2   # tamanho do conjunto Pareto
FEEDBACK_SIZE = 1 # tamanho do minibatch de feedback

dataset = generate_hotpot_pareto_dataset(
    n=N_QUESTIONS,
    seed=42,
    hotpot_path=root_dir / "data" / "hotpot_train_v1.1.json",
)

print(f"Dataset gerado com {len(dataset)} datapoints.")
print("\nExemplo (datapoint 0):")
print(f"  Questão  : {dataset[0]['question']}")
print(f"  Resposta : {dataset[0]['expected_description']}")

# Limita ao tamanho necessário para o GEPA
dataset = dataset[: PARETO_SIZE + FEEDBACK_SIZE]

## 5. Definição do CompoundAISystem

O módulo `description_building` é o alvo da otimização. O GEPA vai evoluir
seu `prompt` (role / system prompt) ao longo das gerações.

O fluxo de controle `KontexFlowGeneralized` orquestra a conversa entre os
agentes: questioning → description_building → self_critique.

In [ ]:
system = CompoundAISystem(
    modules={
        "description_building": LanguageModule(
            id="hotpot_description_builder",
            prompt=hotpotqa_description_building_role,  # ponto de partida
            model_weights="gpt-5-mini",
        )
    },
    control_flow=KontexFlowGeneralized(),
    input_schema=IOSchema(
        fields={"full_knowledge": FullKnowledge},
        required=["full_knowledge"],
    ),
    output_schema=IOSchema(
        fields={"output": int},
        required=["output"],
    ),
    system_id="kontex_hotpotqa",
)

print("✅ CompoundAISystem criado.")
print(f"   Módulos: {list(system.modules.keys())}")
print(f"   Fluxo  : {type(system.control_flow).__name__}")

## 6. Configuração do GEPA

In [ ]:
gepa_config = GEPAConfig(
    inference=InferenceConfig(
        provider="openai",
        model="gpt-5-mini",
        api_key=env.api_key,
        max_tokens=4096,
        temperature=0.1,
        timeout=30,
        base_url=env.base_url,
        retry_attempts=3,
    ),
    optimization=OptimizationConfig(
        budget=20,
        pareto_set_size=PARETO_SIZE,
        minibatch_size=FEEDBACK_SIZE,
        enable_crossover=True,
        crossover_probability=0.3,
        mutation_types=["rewrite", "insert"],
    ),
    database=DatabaseConfig(url="sqlite:///gepa_hotpotqa.db"),
    observability=ObservabilityConfig(
        log_level="INFO",
        log_file="gepa_hotpotqa.log",
        enable_logging=True,
    ),
)

print("✅ GEPAConfig criado.")
print(f"   Budget      : {gepa_config.optimization.budget}")
print(f"   Pareto size : {gepa_config.optimization.pareto_set_size}")
print(f"   Mutations   : {gepa_config.optimization.mutation_types}")

## 7. Avaliador e cliente de inferência

## 7. Critérios de avaliação (GEval)

Edite as strings abaixo para personalizar como o `GEvalMetric` julga as respostas.
O padrão `GEVAL_CRITERIA_HOTPOTQA` já está carregado; basta sobrescrever se necessário.

In [ ]:
# Ponto de configuração: edite aqui para adaptar a avaliação ao seu experimento.
# Use GEVAL_CRITERIA_HOTPOTQA (perguntas multi-hop) ou GEVAL_CRITERIA_TABLE
# (descrição de tabelas), ou escreva critérios próprios.

FACTUAL_ACCURACY_CRITERIA = GEVAL_CRITERIA_HOTPOTQA["factual_accuracy"]
COMPLETENESS_CRITERIA     = GEVAL_CRITERIA_HOTPOTQA["completeness"]

print("Critério — Acurácia Factual:")
print(FACTUAL_ACCURACY_CRITERIA)
print()
print("Critério — Completude:")
print(COMPLETENESS_CRITERIA)

In [ ]:
evaluator = SimpleFeedbackEvaluator([
    GEvalMetric(
        name="geval_metric",
        factual_accuracy_criteria=FACTUAL_ACCURACY_CRITERIA,
        completeness_criteria=COMPLETENESS_CRITERIA,
    )
])
inference_client = InferenceFactory.create_client(gepa_config.inference)

print("✅ Avaliador e cliente de inferência prontos.")

## 8. Execução da otimização GEPA

In [ ]:
import traceback

optimizer = GEPAOptimizer(
    config=gepa_config,
    evaluator=evaluator,
    inference_client=inference_client,
)

result = None
try:
    result = await optimizer.optimize(system, dataset, max_generations=5)
    print("✅ Otimização concluída!")
    print(f"   Melhor score    : {result.best_score:.3f}")
    print(f"   Total rollouts  : {result.total_rollouts}")
    print(f"   Custo total     : ${result.total_cost:.4f}")
    print(f"   Fronteira Pareto: {result.pareto_frontier.size()} pontos")
except Exception:
    print("❌ Otimização falhou:")
    print(traceback.format_exc())
finally:
    if hasattr(inference_client, "close"):
        await inference_client.close()

## 9. Resultados e salvamento do prompt otimizado

In [ ]:
if result is not None:
    from datetime import datetime
    best_module = result.best_system.modules["description_building"]
    print("🧠 Role otimizado do description_building:")
    print("-" * 50)
    print(best_module.prompt)
    print("-" * 50)

    # Salva em disco
    out_dir = root_dir / "optimized_prompts"
    out_dir.mkdir(exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_file = out_dir / f"description_building_optimized_{timestamp}.txt"
    out_file.write_text(best_module.prompt)
    print(f"\n💾 Prompt salvo em: {out_file}")

    # Estatísticas
    stats = optimizer.get_statistics()
    print("\n📊 Estatísticas:")
    print(f"   Gerações       : {stats.get('generations', 0)}")
    print(f"   Mutações ok    : {stats.get('successful_mutations', 0)}")
    print(f"   Melhoria média : {stats.get('average_improvement', 0):.3f}")
else:
    print("Nenhum resultado disponível (otimização falhou).")